In [1]:
import pandas as pd
from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from dotenv import load_dotenv
import sys
import os
project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

In [2]:
load_dotenv()

BASE_PATH = os.getenv("BASE_PATH")
X_train = pd.read_parquet(f"{BASE_PATH}/data/final/X_train_final.parquet")
X_test = pd.read_parquet(f"{BASE_PATH}/data/final/X_test_final.parquet")
y_train = pd.read_parquet(f"{BASE_PATH}/data/final/y_train_final.parquet")
y_test = pd.read_parquet(f"{BASE_PATH}/data/final/y_test_final.parquet")

In [3]:
y_train = y_train.iloc[:, 0]
y_test = y_test.iloc[:, 0]

In [4]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(y_train.dtypes, type(y_train))

(79588, 126) (19898, 126) (79588,) (19898,)
bool <class 'pandas.Series'>


In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
for col in [X_tr, X_val, y_tr, y_val]:
    print(col.shape)

(63670, 126)
(15918, 126)
(63670,)
(15918,)


In [6]:
neg, pos = y_tr.value_counts()[False], y_tr.value_counts()[True]
scale_pos_weight = neg / pos

print(f"Negative (no rain): {neg}, Positive (rain): {pos}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

Negative (no rain): 49362, Positive (rain): 14308
scale_pos_weight: 3.4500


In [7]:
model = lgb.LGBMClassifier(
    objective='binary',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_estimators=1000,
    metric='average_precision'
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='average_precision',
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=50)]
)

/home/youssef/Projects/aus-weather-analysis/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 14308, number of negative: 49362
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015495 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5302
[LightGBM] [Info] Number of data points in the train set: 63670, number of used features: 126
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.224721 -> initscore=-1.238362
[LightGBM] [Info] Start training from score -1.238362
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.727803
[100]	valid_0's average_precision: 0.73925
[150]	valid_0's average_precision: 0.741709
[200]	valid_0's average_precision: 0.743315
[250]	valid_0's average_precision: 0.744584
[300]	valid_0's average_precision: 0.745572
[350]	valid_0's average_precision: 0.745171
Early stopping, best iteration is:
[302]	valid_0's average_precision: 0.74

,n_estimators,1000
,objective,'binary'
,random_state,42
,scale_pos_weight,np.float64(3.449958065417948)
,metric,'average_precision'
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,subsample_for_bin,200000
,class_weight,None


In [8]:
y_pred_proba = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"PR-AUC: {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

PR-AUC: 0.7553
ROC-AUC: 0.8933


In [9]:
import joblib
joblib.dump(model, f"{BASE_PATH}/models/lgbm_baseline.pkl")

['/home/youssef/Projects/aus-weather-analysis/models/lgbm_baseline.pkl']

In [10]:
from src.utils import load_and_split_data
from src.tune_model import tune_model
import optuna.visualization as vis

# Step 1: load + split (reuses baseline's split, keeps X_test untouched)
X_tr, X_val, y_tr, y_val, X_train, y_train, X_test, y_test, scale_pos_weight = load_and_split_data()

# Step 2: run tuning
study = tune_model(X_tr, y_tr, X_val, y_val, scale_pos_weight, n_trials=75)
# Step 3: inspect results
print(f"Best PR-AUC (validation): {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

# Step 4: visualize
vis.plot_param_importances(study).show()
vis.plot_optimization_history(study).show()

[I 2026-08-04 11:50:05,829] A new study created in memory with name: no-name-1a103e14-5753-48c7-8078-ca1b2cb669ba


  0%|          | 0/75 [00:00<?, ?it/s]

/home/youssef/Projects/aus-weather-analysis/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-04 11:50:06,804] Trial 0 finished with value: 0.7386315478426547 and parameters: {'num_leaves': 105, 'learning_rate': 0.2536999076681772, 'max_depth': 10, 'min_child_samples': 62, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 0.7386315478426547.


/home/youssef/Projects/aus-weather-analysis/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

[I 2026-08-04 11:56:24,424] Trial 72 finished with value: 0.7529371961173925 and parameters: {'num_leaves': 96, 'learning_rate': 0.03836631083941031, 'max_depth': 8, 'min_child_samples': 97, 'subsample': 0.6801265310736001, 'colsample_bytree': 0.5543630739330362, 'reg_alpha': 7.150660809018745e-07, 'reg_lambda': 0.4439345197539518}. Best is trial 69 with value: 0.754370888284997.


/home/youssef/Projects/aus-weather-analysis/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-04 11:56:30,618] Trial 73 finished with value: 0.7529466380101613 and parameters: {'num_leaves': 107, 'learning_rate': 0.02515428523749103, 'max_depth': 9, 'min_child_samples': 98, 'subsample': 0.6530621274831792, 'colsample_bytree': 0.6169583998251751, 'reg_alpha': 2.1510933424277636e-05, 'reg_lambda': 3.124610524122173}. Best is trial 69 with value: 0.754370888284997.


/home/youssef/Projects/aus-weather-analysis/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-04 11:56:35,648] Trial 74 finished with value: 0.7526544496202042 and parameters: {'num_leaves': 116, 'learning_rate': 0.03051593292642923, 'max_depth': 7, 'min_child_samples': 90, 'subsample': 0.73576330959542, 'colsample_bytree': 0.5008690976016957, 'reg_alpha': 0.004330882673471398, 'reg_lambda': 0.08029218723478493}. Best is trial 69 with value: 0.754370888284997.
Best PR-AUC (validation): 0.7544
Best params: {'num_leaves': 125, 'learning_rate': 0.03587496887467129, 'max_depth': 9, 'min_child_samples': 97, 'subsample': 0.6539775450229475, 'colsample_bytree': 0.5355248480415369, 'reg_alpha': 2.665982633824293e-05, 'reg_lambda': 0.9681044366445308}
Best PR-AUC (validation): 0.7544
Best params: {'num_leaves': 125, 'learning_rate': 0.03587496887467129, 'max_depth': 9, 'min_child_samples': 97, 'subsample': 0.6539775450229475, 'colsample_bytree': 0.5355248480415369, 'reg_alpha': 2.665982633824293e-05, 'reg_lambda': 0.9681044366445308}


In [11]:
import joblib as job
from dotenv import load_dotenv
import os
from src.utils import load_and_split_data

load_dotenv()
BASE_PATH = os.getenv("BASE_PATH")

clf = job.load(f"{BASE_PATH}/models/lgbm_tuned.pkl")
X_tr, X_val, y_tr, y_val, X_train, y_train, X_test, y_test, scale_pos_weight = load_and_split_data()

In [12]:
import pandas as pd

importances = pd.DataFrame({
    'feature': X_tr.columns,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 most important:")
print(importances.head(10))

print("\nBottom 10 least important:")
print(importances.tail(10))

print("\nRainToday / RainfallLog importance:")
print(importances[importances['feature'].isin(['RainToday', 'RainfallLog'])])

Top 10 most important:
                     feature  importance
120             PressureDiff        2842
11               Pressure3pm        2317
114                TempRange        2288
115               TempChange        2242
119  TempHumidityInteraction        2178
4                   Sunshine        2172
117             HumidityDiff        2020
0                    MinTemp        1991
5              WindGustSpeed        1944
3                Evaporation        1885

Bottom 10 least important:
                  feature  importance
42          Location_Nhil          28
99         WindDir3pm_ENE          25
61      Location_Watsonia          23
96         WindDir9am_WNW          23
41     Location_Newcastle          22
20  Location_AliceSprings          22
37       Location_Mildura          21
45     Location_Nuriootpa          17
38         Location_Moree          15
58         Location_Uluru           2

RainToday / RainfallLog importance:
         feature  importance
125  RainfallL

In [13]:
import joblib as job
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from dotenv import load_dotenv
import os
from src.utils import load_and_split_data

load_dotenv()
BASE_PATH = os.getenv("BASE_PATH")

# Load both models
baseline = job.load(f"{BASE_PATH}/models/lgbm_baseline.pkl")
tuned = job.load(f"{BASE_PATH}/models/lgbm_tuned.pkl")

# Load test set (same for both, unchanged)
_, _, _, _, _, _, X_test, y_test, _ = load_and_split_data()

results = []
for name, model in [('Baseline', baseline), ('Tuned', tuned)]:
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'PR-AUC': average_precision_score(y_test, y_pred_proba),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
        'Best Iteration': model.best_iteration_,
        'Non-zero Features': (model.feature_importances_ > 0).sum(),
    })

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))

   Model   PR-AUC  ROC-AUC  Best Iteration  Non-zero Features
Baseline 0.755301 0.893310             302                124
   Tuned 0.761270 0.896231             730                126
